In [56]:

import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, Baseline
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import SMAPE
from pytorch_lightning import Trainer, LightningModule

# Assuming the dataset is loaded into a pandas DataFrame named df
df = pd.read_parquet('sales_features.parquet')

df = df.reset_index(drop=True)

# Define the maximum prediction length and maximum encoder length
max_prediction_length = 30
max_encoder_length = 60

# Define the training cut-off date
training_cutoff = df["date"].max() - pd.Timedelta(days=max_prediction_length)

categorical = ["day_of_week", "month", "quarter", "year", "is_weekend", "day_of_month", "week_of_year", "is_holiday", "holiday_name", "is_day_before_holiday", "is_day_after_holiday"]

for ca in categorical:
    df[ca] = df[ca].astype(str)

df['group_id'] = df.groupby(['name', 'address', 'zipcode']).ngroup().astype(str)
df['time_idx'] = df.groupby('group_id')['date'].transform(lambda x: (x - x.min()).dt.days)


df = df.drop_duplicates(subset=['group_id', 'time_idx'])
df = df.dropna()

# Create the TimeSeriesDataSet object
training = TimeSeriesDataSet(
    df[lambda x: x.date <= training_cutoff],
    time_idx="time_idx",
    target="sale_dollars",
    group_ids=["group_id"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["name", "address", "city", "county"],
    static_reals=["zipcode", "lon", "lat"],
    time_varying_known_categoricals=categorical,
    time_varying_known_reals=["days_to_nearest_holiday"],
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=[
        "sale_bottles", "sale_bottles_mean", "sale_bottles_median", "sale_bottles_std", "sale_bottles_min",
        "sale_bottles_max", "sale_bottles_skew", "sale_bottles_sem", "sale_dollars_mean", "sale_dollars_median",
        "sale_dollars_std", "sale_dollars_min", "sale_dollars_max", "sale_dollars_skew", "sale_dollars_sem",
        "sale_liters", "sale_liters_mean", "sale_liters_median", "sale_liters_std", "sale_liters_min",
        "sale_liters_max", "sale_liters_skew", "sale_liters_sem", "sale_gallons", "sale_gallons_mean",
        "sale_gallons_median", "sale_gallons_std", "sale_gallons_min", "sale_gallons_max",
        # Add all other relevant columns here
    ],
    target_normalizer=GroupNormalizer(groups=["name"]),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)

train_groups = training.data['groups'].unique()
validation_df = df[df['group_id'].isin(train_groups)].reset_index(drop=True)
validation = TimeSeriesDataSet.from_dataset(training, validation_df, predict=True, stop_randomization=True)

# Create dataloaders for model
batch_size = 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=0)


# Define the TemporalFusionTransformer model
class TFTModel(LightningModule):
    def __init__(self, training):
        super().__init__()
        self.tft = TemporalFusionTransformer.from_dataset(
            training,
            learning_rate=0.03,
            hidden_size=16,
            attention_head_size=1,
            dropout=0.1,
            hidden_continuous_size=8,
            output_size=7,
            loss=SMAPE(),
            log_interval=10,
            reduce_on_plateau_patience=4,
        )
    
    def forward(self, x):
        return self.tft(x)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.tft.loss(y_hat, y)
        return loss
    
    def configure_optimizers(self):
        return self.tft.configure_optimizers()

model = TFTModel(training)


# Define the TemporalFusionTransformer model
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=8,
    output_size=7,
    loss=SMAPE(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)

# Train the model
trainer = Trainer(
    max_epochs=30
)

trainer.fit(
    model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

# Save the model
tft.save_model("tft_model.pth")

print("Model training complete and saved as tft_model.pth")


ValueError: cannot set a frame with no defined index and a scalar